# TAMER OCR — Minimal Runner
Orchestrates environment, pulls code, and launches training.

In [ ]:
import os, shutil, getpass, sys

print("🧹 Purging environment...")
REPO_DIR = 'AI_PROJECT_TAMER_Complete'
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)

IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
os.chdir(WORK_DIR)

print("🔑 Authentication...")
hf_token = os.getenv('HF_TOKEN') or getpass.getpass('HuggingFace Token: ')
os.environ['HF_TOKEN'] = hf_token

kaggle_user = 'merselfares'
kaggle_key = os.getenv('KAGGLE_KEY') or getpass.getpass('Kaggle Key: ')
os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = kaggle_user, kaggle_key
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f: 
    f.write(f'{{"username":"{kaggle_user}","key":"{kaggle_key}"}}')
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print("📥 Pulling codebase...")
!pip install -q timm albumentations datasets huggingface_hub requests tqdm psutil opencv-python-headless
!git clone https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete

sys.path.insert(0, os.path.join(WORK_DIR, REPO_DIR))
print("✅ Setup complete.")

In [ ]:
from tamer_ocr.config import Config
from tamer_ocr.core.trainer import Trainer
from huggingface_hub import HfApi
from huggingface_hub import hf_hub_download

config = Config()
config.data_dir = os.path.join(WORK_DIR, 'data')
config.output_dir = os.path.join(WORK_DIR, 'outputs')
config.checkpoint_dir = os.path.join(WORK_DIR, 'checkpoints')

for d in [config.data_dir, config.output_dir, config.checkpoint_dir]: 
    os.makedirs(d, exist_ok=True)

hf_user = HfApi(token=os.environ['HF_TOKEN']).whoami()['name']
config.hf_token = os.environ['HF_TOKEN']
config.hf_repo_id = f"{hf_user}/tamer-ocr-model"
config.hf_dataset_repo_id = f"{hf_user}/tamer-preprocessed"

print("🚀 Launching Pipeline...")
trainer = Trainer(config)
trainer.run()

In [ ]:
from tamer_ocr.utils.checkpoint import push_checkpoint_to_hf
best_model = os.path.join(config.checkpoint_dir, 'best.pt')
if os.path.exists(best_model):
    push_checkpoint_to_hf(best_model, config, epoch=0, is_best=True)
    print(f"✅ Pushed best model to {config.hf_repo_id}")